In [3]:
import time
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException

def scrape_srilankan_flights(url):
    # Initialize Chrome
    options = webdriver.ChromeOptions()
    # options.add_argument('--headless') # Uncomment to run in background
    driver = webdriver.Chrome(options=options)
    
    print(f"Opening URL: {url}")
    driver.get(url)
    
    # --- STEP 1: Handle the "View more" Button ---
    while True:
        try:
            # Look for a button containing the text 'View more' (case-insensitive)
            view_more_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'view more')]"))
            )
            
            # Scroll to the button
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", view_more_btn)
            time.sleep(1) 
            
            print("Clicking 'View more'...")
            view_more_btn.click()
            time.sleep(3) # Wait for new cards to render
            
        except TimeoutException:
            print("No more 'View more' button found. All flights loaded.")
            break
        except ElementClickInterceptedException:
            print("Button click intercepted. Trying via JavaScript...")
            driver.execute_script("arguments[0].click();", view_more_btn)
            time.sleep(3)
        except Exception as e:
            print(f"Finished loading pages. Reason: {e}")
            break

    # --- STEP 2: Extract the Data ---
    print("\nExtracting flight data...")
    flight_data = []
    
    # Find all flight cards based on the screenshot's HTML
    cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-test="card-container"]')
    
    for card in cards:
        try:
            origin_elem = card.find_element(By.CSS_SELECTOR, 'p[data-test="origin-text"]')
            destination_elem = card.find_element(By.CSS_SELECTOR, 'span[data-test="destination-text"]')
            dates_elem = card.find_element(By.CSS_SELECTOR, 'div[data-test="departing-text"]')
            price_elem = card.find_element(By.CSS_SELECTOR, 'div[data-test="price"]')
            last_seen_elem = card.find_element(By.CSS_SELECTOR, 'div[data-test="last-seen"]')
            
            # Clean and store the text
            flight = {
                "Origin": origin_elem.text.replace(destination_elem.text, '').replace('to', '').strip(),
                "Destination": destination_elem.text.strip(),
                "Travel Dates": dates_elem.text.strip(),
                "Price": price_elem.text.strip(),
                "Last Seen": last_seen_elem.text.strip()
            }
            flight_data.append(flight)
            
        except NoSuchElementException:
            continue

    driver.quit()
    return flight_data

# --- STEP 3: RUN SCRIPT & SAVE TO CSV ---
if __name__ == "__main__":
    target_url = "https://www.srilankan.com/en-lk/" 
    scraped_flights = scrape_srilankan_flights(target_url)
    
    if scraped_flights:
        csv_filename = "srilankan_flight_deals.csv"
        
        # Define the column headers based on our dictionary keys
        headers = ["Origin", "Destination", "Travel Dates", "Price", "Last Seen"]
        
        print(f"\nSaving {len(scraped_flights)} flights to {csv_filename}...")
        
        # Write to CSV file
        with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=headers)
            writer.writeheader()
            writer.writerows(scraped_flights)
            
        print("Data successfully saved!")
    else:
        print("No data was scraped to save.")

Opening URL: https://www.srilankan.com/en-lk/
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Clicking 'View more'...
Button click intercepted. Trying via JavaScript...
Clicking 'View more'...
Button click intercepted. Trying via JavaScript...
Clicking 'View more'...
Button click intercepted. Trying via JavaScript...
Clicking 'View more'...
Button click intercepted. Trying via JavaScript...
No more 'View more' button found. All flights loaded.

Extracting flight data...

Saving 171 flights to srilankan_flight_deals.csv...
Data successfully saved!


In [6]:
import pandas as pd
df = pd.read_csv("srilankan_flight_deals.csv")
df.tail()

,Origin,Destination,Travel Dates,Price,Last Seen
166,Colombo (CMB),Dhaka (DAC),04/06/2026 - 04/10/2026,"LKR160,676\n*",Seen: 23 hrs ago
167,Colombo (CMB),Dhaka (DAC),04/06/2026 - 04/23/2026,"LKR160,676\n*",Seen: 23 hrs ago
168,Colombo (CMB),Dhaka (DAC),04/06/2026 - 05/06/2026,"LKR160,676\n*",Seen: 23 hrs ago
169,Colombo (CMB),Dhaka (DAC),04/06/2026 - 04/09/2026,"LKR160,676\n*",Seen: 23 hrs ago
170,Colombo (CMB),Dhaka (DAC),04/06/2026 - 04/16/2026,"LKR160,676\n*",Seen: 23 hrs ago
